# Forecasting German Electricity Demand - An Ensemble-Tree Deep Dive (Version 5)

This is the fifth and final study in the series. It forecasts German national electricity
load two years ahead on a weekly grid, moving through data preparation, exploratory and
seasonality analysis, stationarity testing, a family of naive baselines, a **SARIMA** model
and its exogenous extension **SARIMAX**, a **tuned Random Forest** as the feature-based
learner, and finally an hourly **LSTM**. Every model is scored on one shared 104-week
hold-out with RMSE, MAE and MAPE.

**What is different in this version**
- The feature-based learner is again a **Random Forest**, but treated far more thoroughly
  than a single fixed fit. The ensemble is deliberately **grown larger and deeper** - many
  more trees and richer branching - and its hyper-parameters (tree count, depth, leaf/split
  sizes and the feature-sampling rule) are searched with a **randomised, time-aware
  cross-validation** rather than fixed by hand.
- The bootstrap nature of the forest is used directly: an **out-of-bag** learning curve
  shows how skill responds to the number of trees, and the **spread across individual trees**
  is turned into a prediction-interval fan around the recursive forecast.
- Feature relevance is read two ways - the forest's built-in **impurity** importance and a
  hold-out **permutation** importance - so a single ranking cannot mislead.
- Seasonality is separated with **STL**; the SARIMA order is chosen **residual-adequacy
  first** (a low AIC never overrides a failed white-noise check); temperature is pulled from
  Open-Meteo and **cached locally** so the notebook is reproducible offline.


## Step 1 - Environment, palette and scoring helpers
A compact set-up cell fixes the plotting identity, the random seed and two scoring helpers
reused throughout.

In [ ]:
!pip install holidays --quiet
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# ---- fifth-version visual identity ----
plt.style.use('bmh')
TONE = {
    'forest': '#2d6a4f',
    'clay':   '#bc6c25',
    'steel':  '#457b9d',
    'plum':   '#6a4c93',
    'sand':   '#e9c46a',
    'ash':    '#6c757d',
}
plt.rcParams['axes.prop_cycle'] = plt.cycler(
    color=[TONE['forest'], TONE['clay'], TONE['steel'], TONE['plum'], TONE['sand'], TONE['ash']])
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = '#f7f7f4'
plt.rcParams['font.size'] = 10
RS = 19
np.random.seed(RS)
CYCLE = 52          # weeks per year
LEAD = 104          # forecast horizon in weeks (two years)

def grade(actual, guess):
    """RMSE, MAE and MAPE for one forecast, returned as a dict."""
    actual = np.asarray(actual, float)
    guess = np.asarray(guess, float)
    err = actual - guess
    return {
        'RMSE': float(np.sqrt(np.mean(err ** 2))),
        'MAE': float(np.mean(np.abs(err))),
        'MAPE': float(np.mean(np.abs(err / actual)) * 100.0),
    }

def rmse_val(actual, guess):
    """Stand-alone RMSE used by the neural-network section."""
    actual = np.asarray(actual, float)
    guess = np.asarray(guess, float)
    return float(np.sqrt(np.mean((actual - guess) ** 2)))

## Step 2 - Load the raw series
We read the Open Power System Data 60-minute file. The path is resolved from a short list of
candidate locations so the notebook runs on Kaggle or locally without edits.

In [ ]:
import os
# Portable data location: try the Kaggle mount first, then two local paths. Point
# DATA_LOCATIONS at your copy of the OPSD 60-minute file
# (https://data.open-power-system-data.org/time_series/) if it lives elsewhere.
DATA_LOCATIONS = [
    '/kaggle/input/datasets/rishiande/german/opsd_60min_raw.csv',
    'opsd_60min_raw.csv',
    'data/opsd_60min_raw.csv',
]
DATA_FILE = next((p for p in DATA_LOCATIONS if os.path.exists(p)), None)
if DATA_FILE is None:
    raise FileNotFoundError(
        'opsd_60min_raw.csv was not found. Attach the OPSD 60-minute dataset or add its '
        'path to DATA_LOCATIONS.')
raw_tbl = pd.read_csv(DATA_FILE, parse_dates=['utc_timestamp'], index_col='utc_timestamp')
print(f'Rows read: {raw_tbl.shape[0]:,}  (source: {DATA_FILE})')

## Step 3 - Keep German load and set the study window
The German actual-load column is isolated and the series is clipped to 2015-2020.

In [ ]:
SIGNAL = 'DE_load_actual_entsoe_transparency'
de_load = raw_tbl[[SIGNAL]].rename(columns={SIGNAL: 'mw'}).copy()
# The load column ends 2020-09-30; the 2020-10-31 upper bound is only a harmless slice
# ceiling (.loc stops at the last available row) - not an accidental mid-series cut-off.
de_load = de_load.loc['2015-01-01':'2020-10-31'].dropna()
print('Span        :', de_load.index.min(), '->', de_load.index.max())
print('Hourly rows :', f'{len(de_load):,}')

## Step 4 - Aggregate to daily and weekly means
Weekly means are the modelling grid; the hourly signal is kept for the LSTM in Step 10.

In [ ]:
mw_hour = de_load['mw']
mw_day = mw_hour.resample('D').mean()
mw_week = mw_hour.resample('W').mean()
print('Weekly points :', mw_week.size, '| gaps:', bool(mw_week.isna().any()))
print(mw_week.describe().round(1).to_string())

## Step 5 - Exploratory view
Two lenses: the daily series with a rolling trend, and a month-by-year heatmap that exposes
the annual shape and the 2020 anomaly at a glance.

In [ ]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=(15, 5),
                               gridspec_kw={'width_ratios': [1.35, 1]})
axL.plot(mw_day.index, mw_day, color=TONE['ash'], lw=0.6, alpha=0.7, label='Daily mean')
axL.plot(mw_day.rolling(60, center=True).mean(), color=TONE['forest'], lw=2.3,
         label='60-day rolling mean')
axL.set_title('Daily electricity load with trend')
axL.set_ylabel('MW')
axL.legend(fontsize=8)

# month x year heatmap of weekly load
hm = pd.DataFrame({'mw': mw_week, 'yr': mw_week.index.year, 'mo': mw_week.index.month})
grid = hm.pivot_table(index='mo', columns='yr', values='mw', aggfunc='mean')
im = axR.imshow(grid.values, aspect='auto', cmap='YlGnBu', origin='lower')
axR.set_xticks(range(grid.shape[1]))
axR.set_xticklabels(grid.columns, fontsize=8)
axR.set_yticks(range(12))
axR.set_yticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                     'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], fontsize=8)
axR.set_title('Mean weekly load by month and year')
fig.colorbar(im, ax=axR, fraction=0.046, pad=0.04, label='MW')
plt.tight_layout()
plt.show()

## Step 6 - STL decomposition
STL splits the weekly series into trend, an annual seasonal shape and a remainder. A seasonal
strength close to 1 confirms the yearly cycle dominates.

In [ ]:
from statsmodels.tsa.seasonal import STL
stl = STL(mw_week, period=CYCLE, robust=True).fit()
strength = max(0.0, 1.0 - stl.resid.var() / (stl.resid + stl.seasonal).var())
fig, panes = plt.subplots(3, 1, figsize=(13, 7), sharex=True)
panes[0].plot(mw_week.index, mw_week, color=TONE['ash'], lw=1.0, label='Observed')
panes[0].plot(stl.trend.index, stl.trend, color=TONE['forest'], lw=2.0, label='Trend')
panes[0].legend(loc='upper right', fontsize=8); panes[0].set_ylabel('MW')
panes[1].plot(stl.seasonal.index, stl.seasonal, color=TONE['steel'], lw=1.0)
panes[1].set_ylabel('Seasonal')
panes[2].plot(stl.resid.index, stl.resid, color=TONE['clay'], lw=0.9)
panes[2].axhline(0, color=TONE['ash'], lw=0.8); panes[2].set_ylabel('Remainder')
panes[2].set_xlabel('Date')
panes[0].set_title(f'STL decomposition of weekly load (seasonal strength = {strength:.3f})')
plt.tight_layout()
plt.show()

## Step 7 - Stationarity and correlation structure
ADF and KPSS are read together (they answer opposite null hypotheses), and the ACF/PACF of the
level and first difference guide the SARIMA search that follows.

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
def unit_root_line(series, tag):
    series = series.dropna()
    p_adf = adfuller(series)[1]
    p_kpss = kpss(series, regression='c', nlags='auto')[1]
    print(f'{tag:30s} ADF p={p_adf:6.4f} ({"stationary" if p_adf <= 0.05 else "unit root"})'
          f'  |  KPSS p={p_kpss:6.4f} ({"non-stationary" if p_kpss < 0.05 else "stationary"})')
print('Stationarity checks'); print('-' * 90)
unit_root_line(mw_week, 'level')
unit_root_line(mw_week.diff(), 'first difference')
unit_root_line(mw_week.diff(CYCLE), 'seasonal difference (52)')

fig, gx = plt.subplots(2, 2, figsize=(13.5, 7))
plot_acf(mw_week.dropna(), ax=gx[0, 0], lags=104, title='ACF - level')
plot_pacf(mw_week.dropna(), ax=gx[0, 1], lags=52, method='ywm', title='PACF - level')
plot_acf(mw_week.diff().dropna(), ax=gx[1, 0], lags=104, title='ACF - first difference')
plot_pacf(mw_week.diff().dropna(), ax=gx[1, 1], lags=52, method='ywm', title='PACF - first difference')
plt.tight_layout()
plt.show()

## Step 8 - Train / hold-out split
The final 104 weeks (two years) are withheld from every model and scored identically.

In [ ]:
fit_w = mw_week.iloc[:-LEAD]
oos_w = mw_week.iloc[-LEAD:]
oos_idx = oos_w.index
print(f'Train   : {fit_w.size} weeks (ends {fit_w.index[-1].date()})')
print(f'Hold-out: {oos_w.size} weeks (ends {oos_w.index[-1].date()})')

## Step 9 - Naive benchmarks
Four reference forecasts set the bar every model must clear: the training **Mean**, a **Naive**
carry-forward of the last value, a **Seasonal naive** replay of the last year, and a linear
**Drift**.

In [ ]:
tail_year = fit_w.iloc[-CYCLE:].to_numpy()
step = (fit_w.iloc[-1] - fit_w.iloc[0]) / (fit_w.size - 1)
naive_set = {
    'Mean': pd.Series(fit_w.mean(), index=oos_idx),
    'Naive': pd.Series(fit_w.iloc[-1], index=oos_idx),
    'Seasonal naive': pd.Series(np.take(tail_year, np.arange(LEAD) % CYCLE), index=oos_idx),
    'Drift': pd.Series(fit_w.iloc[-1] + step * np.arange(1, LEAD + 1), index=oos_idx),
}
print('Benchmark scores on the hold-out')
for tag, path in naive_set.items():
    s = grade(oos_w, path)
    print(f"  {tag:16s} RMSE={s['RMSE']:8.1f}  MAE={s['MAE']:8.1f}  MAPE={s['MAPE']:5.2f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.8))
ax.plot(oos_idx, oos_w, color=TONE['forest'], lw=2.4, label='Actual (hold-out)')
styling = {'Seasonal naive': dict(color=TONE['clay'], lw=1.8),
           'Mean': dict(color=TONE['steel'], ls=(0, (4, 2))),
           'Drift': dict(color=TONE['plum'], ls=(0, (1, 1))),
           'Naive': dict(color=TONE['ash'], ls=(0, (5, 1)))}
for tag, kw in styling.items():
    ax.plot(oos_idx, naive_set[tag], label=tag, **kw)
ax.set_title('Benchmark forecasts over the two-year hold-out')
ax.set_ylabel('MW'); ax.legend(ncol=3, fontsize=8)
plt.tight_layout()
plt.show()

## Step 10 - SARIMA order search (screening pass)
A wide grid over the non-seasonal orders is screened quickly with a fast simple-differencing
fit. The screening AIC only ranks orders that share the same `d`; cross-`d` comparisons are
made later with exact refits.

In [ ]:
import itertools
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from joblib import Parallel, delayed
BASE_SEAS = (1, 1, 1, CYCLE)
OFF = dict(enforce_stationarity=False, enforce_invertibility=False)
pq_grid = list(itertools.product(range(7), range(3), range(7)))
print('Non-seasonal orders to screen:', len(pq_grid))

def sift_aic(order, y):
    """Fast simple-differencing fit; AIC is comparable only within a shared d."""
    try:
        res = SARIMAX(y, order=order, seasonal_order=BASE_SEAS, simple_differencing=True,
                      **OFF).fit(disp=False, method='lbfgs', maxiter=45)
        return {'order': order, 'd': order[1], 'aic': res.aic,
                'ok': bool((res.mle_retvals or {}).get('converged', False))}
    except Exception:
        return {'order': order, 'd': order[1], 'aic': np.inf, 'ok': False}

sifted = pd.DataFrame(Parallel(n_jobs=-1)(delayed(sift_aic)(o, fit_w) for o in pq_grid))
low = sifted.sort_values('aic').iloc[0]
print('Raw grid minimum:', tuple(low['order']),
      f"(AIC={low['aic']:.2f}, d={int(low['order'][1])}) - not comparable across d")

## Step 11 - Exact refits with residual adequacy
AIC alone is not a valid criterion: a low-AIC order whose residuals are still autocorrelated
fails the diagnostic requirement. Candidate orders are refit exactly at **both `d = 0` and
`d = 1`** (AIC is only comparable within a fixed `d`) and a Ljung-Box p-value is recorded for
each, so the next cell can select on residual adequacy first.

In [ ]:
D_TRY = (0, 1)
WHITE_P = 0.05          # a Ljung-Box p above this marks the residuals as effectively white noise

def whiten(fitted, order, seasonal):
    """Return warm-up-trimmed standardized residuals for a fitted state-space model."""
    burn = seasonal[3] * seasonal[1] + order[1]
    try:
        vec = np.asarray(fitted.standardized_forecasts_error)
        vec = vec[0] if vec.ndim > 1 else vec
    except Exception:
        vec = np.asarray(fitted.resid)
    return pd.Series(vec).replace([np.inf, -np.inf], np.nan).iloc[burn:].dropna()

def refit_exact(order, seasonal, y, exog=None):
    """Full ML fit; also reports convergence and a whitened-residual Ljung-Box p-value."""
    res = SARIMAX(y, exog=exog, order=order, seasonal_order=seasonal, **OFF
                  ).fit(disp=False, method='lbfgs', maxiter=320)
    wr = whiten(res, order, seasonal)
    lag = min(20, max(1, len(wr) - 2))
    pv = acorr_ljungbox(wr, lags=[lag], return_df=True)['lb_pvalue'].iloc[0]
    converged = bool((res.mle_retvals or {}).get('converged', False))
    return res, converged, pv

def refits_for_d(d_value, k=9):
    """Exact refits of the k best-screening orders that share this differencing d."""
    seed = (sifted[(sifted['d'] == d_value) & sifted['ok']]
            .sort_values('aic').head(k)['order'].tolist())
    if not seed:
        seed = sifted[sifted['d'] == d_value].sort_values('aic').head(k)['order'].tolist()
    if not seed:
        seed = [(1, d_value, 1), (0, d_value, 1)]
    recs = []
    for od in seed:
        try:
            res, conv, pv = refit_exact(od, BASE_SEAS, fit_w)
            recs.append({'order': od, 'aic': res.aic, 'bic': res.bic, 'conv': conv, 'lb_p': round(pv, 4)})
        except Exception:
            recs.append({'order': od, 'aic': np.inf, 'bic': np.inf, 'conv': False, 'lb_p': np.nan})
    frame = pd.DataFrame(recs)
    return frame[np.isfinite(frame['aic'])].sort_values('aic').reset_index(drop=True)

shelf = {d_value: refits_for_d(d_value) for d_value in D_TRY}
for d_value, frame in shelf.items():
    print(f'Exact refits at d={d_value} (AIC comparable within this d):')
    print(frame.to_string(index=False) if not frame.empty else '  (no convergent fit)')
    print()

## Step 12 - Choose the order: adequacy first, then AIC and parsimony
The rule is deliberate. Keep only orders whose residuals are white noise (Ljung-Box
`p > 0.05`); prefer the **smallest `d`** that already achieves this (which matches the
stationarity evidence and avoids over-differencing); within that `d` take the lowest AIC and
break ties on the fewest AR+MA terms. If no order at any `d` whitens the residuals, the
best-AIC model is kept but flagged openly.

In [ ]:
# Choose on residual adequacy first, then AIC + parsimony. Because AIC cannot be compared
# across differencing orders, we accept the FEWEST differences that already delivers white
# residuals; AIC ties (within 2 units) are then broken toward fewer AR+MA terms. If nothing
# clears the white-noise bar at any d, the lowest-AIC model is retained but clearly labelled.
def elect(shelf):
    for d_value in D_TRY:
        frame = shelf.get(d_value)
        if frame is None or frame.empty:
            continue
        clear = frame[frame['lb_p'] > WHITE_P]
        if not clear.empty:
            return d_value, clear, f'fewest differences giving white residuals (Ljung-Box p > {WHITE_P})'
    backup = shelf.get(1)
    if backup is None or backup.empty:
        backup = next(f for f in shelf.values() if not f.empty)
    return int(backup['order'].iloc[0][1]), backup, 'no white-noise order at any d - kept best AIC (flagged: residuals not white)'

pick_d, keep, why = elect(shelf)
base_aic = keep['aic'].min()
close = keep[keep['aic'] <= base_aic + 2.0]
if close.empty:
    close = keep.head(1)
ar_order = min(close['order'], key=lambda o: (o[0] + o[2], o[0]))
print('Differencing kept  :', pick_d)
print('Why                :', why)
print('White-noise orders :', list(keep['order']))
print('AIC-tie window     :', list(close['order']))
print('Selected order     :', ar_order,
      f"(lb_p={keep.loc[keep['order'] == ar_order, 'lb_p'].iloc[0]})")

## Step 13 - Seasonal order search
With the non-seasonal order fixed, the seasonal `(P, Q)` are searched over `{0, 1}` under the
same rule: prefer white-noise residuals, then the lowest AIC.

In [ ]:
seas_menu = [(P, 1, Q, CYCLE) for P in (0, 1) for Q in (0, 1)]
srec = []
for so in seas_menu:
    try:
        res, conv, pv = refit_exact(ar_order, so, fit_w)
        srec.append({'seasonal': so, 'aic': res.aic, 'bic': res.bic, 'conv': conv, 'lb_p': round(pv, 4)})
    except Exception:
        srec.append({'seasonal': so, 'aic': np.inf, 'bic': np.inf, 'conv': False, 'lb_p': np.nan})
seas_tbl = pd.DataFrame(srec)
seas_tbl = seas_tbl[np.isfinite(seas_tbl['aic'])].sort_values('aic').reset_index(drop=True)
if seas_tbl.empty:
    seas_tbl = pd.DataFrame([{'seasonal': BASE_SEAS, 'aic': np.nan, 'bic': np.nan, 'conv': False, 'lb_p': np.nan}])
print(seas_tbl.to_string(index=False))
seas_clear = seas_tbl[seas_tbl['lb_p'] > WHITE_P]
if not seas_clear.empty:
    seas_order = seas_clear['seasonal'].iloc[0]
    print('\nWhite-noise seasonal orders:', list(seas_clear['seasonal']))
else:
    seas_order = seas_tbl['seasonal'].iloc[0]
    print('\nNo seasonal order gave white residuals; kept best AIC.')
print('Chosen seasonal order:', seas_order)

## Step 14 - Fit the final SARIMA and inspect residuals
The chosen model is refit and its whitened residuals are checked for normality (Shapiro-Wilk)
and remaining autocorrelation (Ljung-Box at several lags).

In [ ]:
import scipy.stats as sstats
sarima_fit = SARIMAX(fit_w, order=ar_order, seasonal_order=seas_order, **OFF
                     ).fit(disp=False, method='lbfgs', maxiter=450)
whitened = whiten(sarima_fit, ar_order, seas_order)
p_shapiro = sstats.shapiro(whitened)[1]
lb_lags = [l for l in (10, 20, 52) if l < len(whitened)]
lb_tbl = acorr_ljungbox(whitened, lags=lb_lags, return_df=True)
line = '=' * 58
print(line)
print('FINAL SARIMA  {} x {}'.format(ar_order, seas_order))
print(line)
print('AIC / BIC       : {:.2f} / {:.2f}'.format(sarima_fit.aic, sarima_fit.bic))
print('Converged       :', bool(sarima_fit.mle_retvals.get('converged', False)))
print('Shapiro-Wilk p  : {:.3f} ({})'.format(p_shapiro, 'normal' if p_shapiro > 0.05 else 'non-normal'))
print('Ljung-Box on whitened residuals:')
print(lb_tbl.to_string())

In [ ]:
# Manual diagnostics (plot_diagnostics can crash on short seasonal series).
fig, (dxa, dxb) = plt.subplots(1, 2, figsize=(12.6, 4.2))
plot_acf(whitened, ax=dxa, lags=min(52, len(whitened) // 2 - 1), title='Whitened residuals - ACF')
dxb.hist(whitened, bins=24, density=True, color=TONE['steel'], alpha=0.7, edgecolor='white')
xs = np.linspace(float(whitened.min()), float(whitened.max()), 200)
dxb.plot(xs, sstats.norm.pdf(xs), color=TONE['clay'], lw=2, label='N(0, 1)')
dxb.set_title('Whitened residuals - distribution'); dxb.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
sarima_bundle = sarima_fit.get_forecast(steps=LEAD)
sarima_fc = sarima_bundle.predicted_mean
sarima_fc.index = oos_idx
sarima_band = sarima_bundle.conf_int(alpha=0.05)
sarima_band.index = oos_idx
fig, ax = plt.subplots(figsize=(12, 4.8))
ax.plot(fit_w.index[-70:], fit_w.iloc[-70:], color=TONE['ash'], lw=1.1, label='Recent history')
ax.plot(oos_idx, oos_w, color=TONE['forest'], lw=2.2, label='Actual')
ax.plot(oos_idx, sarima_fc, color=TONE['clay'], lw=2.0, ls=(0, (5, 1)), label='SARIMA mean')
ax.fill_between(oos_idx, sarima_band.iloc[:, 0], sarima_band.iloc[:, 1],
                color=TONE['clay'], alpha=0.15, label='95% interval')
ax.set_title('SARIMA {} x {} forecast'.format(ar_order, seas_order))
ax.set_ylabel('MW'); ax.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()
sarima_err = grade(oos_w, sarima_fc)
print('SARIMA  RMSE={RMSE:.1f}  MAE={MAE:.1f}  MAPE={MAPE:.2f}%'.format(**sarima_err))

## Step 15 - Exogenous drivers (temperature + holidays)
Temperature is pulled from the Open-Meteo archive for Berlin and **cached to a local CSV** so
later runs work offline. A German public-holiday flag is added. Both feed SARIMAX here and the
Random Forest later.

In [ ]:
import requests
import holidays
ENDPOINT = 'https://archive-api.open-meteo.com/v1/archive'
WEATHER_CACHE = 'berlin_temperature_2015_2020.csv'
weather_args = {'latitude': 52.52, 'longitude': 13.41, 'start_date': '2015-01-01',
                'end_date': '2020-09-30', 'hourly': 'temperature_2m', 'timezone': 'UTC'}
# Offline-safe: read the cache if present, else call the API once and cache the response.
if os.path.exists(WEATHER_CACHE):
    temp_hz = pd.read_csv(WEATHER_CACHE, parse_dates=['time'], index_col='time')['temperature_2m']
    print(f'Loaded cached temperature from {WEATHER_CACHE}')
else:
    try:
        payload = requests.get(ENDPOINT, params=weather_args, timeout=60).json()
    except Exception as exc:
        raise RuntimeError('Open-Meteo request failed and no local cache '
                           f'({WEATHER_CACHE}) was found. Connect once to build the cache, '
                           'or ship the cache CSV.') from exc
    temp_hz = pd.Series(payload['hourly']['temperature_2m'],
                        index=pd.to_datetime(payload['hourly']['time']), name='temperature_2m')
    temp_hz.index.name = 'time'
    temp_hz.to_csv(WEATHER_CACHE)
    print(f'Fetched temperature from Open-Meteo and cached to {WEATHER_CACHE}')
temp_hz.index = pd.to_datetime(temp_hz.index)
if temp_hz.index.tz is None:
    temp_hz.index = temp_hz.index.tz_localize('UTC')
else:
    temp_hz.index = temp_hz.index.tz_convert('UTC')
temp_week = (temp_hz.resample('W').mean().reindex(mw_week.index).interpolate().bfill().ffill())
cal = holidays.Germany(years=range(2015, 2021))
holiday_week = pd.Series([int(any(d in cal for d in pd.date_range(end=w, periods=7)))
                          for w in mw_week.index], index=mw_week.index)
driver = pd.DataFrame({'temp': temp_week, 'temp_sq': temp_week ** 2,
                       'temp_back': temp_week.shift(1).bfill(), 'holiday': holiday_week})
print('Drivers:', list(driver.columns), '| nulls:', int(driver.isna().sum().sum()))
driver_fit = driver.iloc[:-LEAD]
driver_oos = driver.iloc[-LEAD:]

In [ ]:
# temperature-load relationship as a 2-D density with a quadratic overlay
fig, ax = plt.subplots(figsize=(8.4, 5.2))
hb = ax.hexbin(driver['temp'], mw_week, gridsize=22, cmap='BuGn', mincnt=1)
quad = np.polyfit(driver['temp'], mw_week, 2)
xr = np.linspace(driver['temp'].min(), driver['temp'].max(), 120)
ax.plot(xr, np.polyval(quad, xr), color=TONE['clay'], lw=2.4, label='Quadratic fit')
ax.set_title('Weekly load vs temperature (density)')
ax.set_xlabel('Weekly mean temperature (C)'); ax.set_ylabel('Weekly mean load (MW)')
fig.colorbar(hb, ax=ax, label='weeks per cell'); ax.legend()
plt.tight_layout()
plt.show()

## Step 16 - SARIMAX with temperature and holidays
The exogenous regressors are added to the same order. The temperature forecast over the horizon
is treated as known (a conditional forecast).

In [ ]:
sarimax_fit = SARIMAX(fit_w, exog=driver_fit, order=ar_order, seasonal_order=seas_order, **OFF
                      ).fit(disp=False, method='lbfgs', maxiter=450)
sarimax_bundle = sarimax_fit.get_forecast(steps=LEAD, exog=driver_oos)
sarimax_fc = sarimax_bundle.predicted_mean
sarimax_fc.index = oos_idx
sarimax_band = sarimax_bundle.conf_int(alpha=0.05)
sarimax_band.index = oos_idx
sarimax_err = grade(oos_w, sarimax_fc)
print('SARIMAX RMSE={RMSE:.1f}  MAE={MAE:.1f}  MAPE={MAPE:.2f}%'.format(**sarimax_err))
print('SARIMA  RMSE={:.1f}  (no drivers)'.format(sarima_err['RMSE']))
fig, ax = plt.subplots(figsize=(12, 4.8))
ax.plot(oos_idx, oos_w, color=TONE['forest'], lw=2.2, label='Actual')
ax.plot(oos_idx, sarimax_fc, color=TONE['steel'], lw=2.0, label='SARIMAX mean')
ax.fill_between(oos_idx, sarimax_band.iloc[:, 0], sarimax_band.iloc[:, 1],
                color=TONE['steel'], alpha=0.16, label='95% interval')
ax.set_title('SARIMAX (temperature + holiday) forecast')
ax.set_ylabel('MW'); ax.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

## Step 17 - Feature-based learner: a fuller Random Forest

Version 1 fit a single, hand-sized forest. Here the same estimator family is pushed much
harder. The design matrix mixes calendar terms, temperature (level, square, lag and a
four-week trailing mean), a holiday flag and the load's own lag-1 and lag-52 values. Then:

1. a **randomised search** over a wide space - many more trees and deeper branching than a
   fixed forest - is scored with a **time-aware** cross-validation, so ordering is respected;
2. an **out-of-bag** learning curve shows how error responds to the number of trees, which is
   exactly why the ensemble is grown large;
3. the forecast is produced recursively (a true two-year forecast) and, for reference, one step
   ahead; and
4. the **spread across the individual trees** is turned into a prediction-interval fan.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.inspection import permutation_importance
woy = mw_week.index.isocalendar().week.astype(int).to_numpy()
feat_mat = pd.DataFrame(index=mw_week.index)
feat_mat['woy'] = woy
feat_mat['month'] = mw_week.index.month
feat_mat['temp'] = driver['temp']
feat_mat['temp_sq'] = driver['temp'] ** 2
feat_mat['temp_back'] = driver['temp'].shift(1)
feat_mat['temp_roll4'] = driver['temp'].rolling(4).mean()
feat_mat['holiday'] = driver['holiday']
feat_mat['load_back1'] = mw_week.shift(1)
feat_mat['load_back52'] = mw_week.shift(CYCLE)
feat_mat['y'] = mw_week.to_numpy()
feat_mat = feat_mat.dropna()
PREDICTORS = [c for c in feat_mat.columns if c != 'y']
feat_fit = feat_mat.iloc[:-LEAD]
feat_oos = feat_mat.iloc[-LEAD:]
Xf, yf = feat_fit[PREDICTORS], feat_fit['y']
Xo, yo = feat_oos[PREDICTORS], feat_oos['y']
print('Design matrix:', Xf.shape, '| predictors:', PREDICTORS)

### Step 17a - Randomised, time-aware hyper-parameter search
The search deliberately reaches for **more and deeper trees** than a default forest, and also
tunes leaf/split sizes and the feature-sampling rule. `TimeSeriesSplit` keeps every validation
fold strictly after its training fold, so no future week informs the past.

In [ ]:
rf_space = {
    'n_estimators': [400, 600, 800, 1000, 1200],
    'max_depth': [16, 24, 32, None],
    'min_samples_leaf': [1, 2, 4],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 0.5, 1.0],
}
splitter = TimeSeriesSplit(n_splits=4)
rf_search = RandomizedSearchCV(
    RandomForestRegressor(bootstrap=True, random_state=RS, n_jobs=-1),
    rf_space, n_iter=25, cv=splitter, scoring='neg_root_mean_squared_error',
    random_state=RS, n_jobs=-1)
rf_search.fit(Xf, yf)
best = rf_search.best_params_
print('Cross-validated RMSE :', round(-rf_search.best_score_, 1), 'MW')
print('Best hyper-parameters:')
for k, v in best.items():
    print(f'   {k:18s}: {v}')

### Step 17b - Out-of-bag learning curve
Because the forest bootstraps its training rows, each tree leaves roughly a third of the data
unseen, giving a nearly free validation estimate. Tracking the out-of-bag R^2 as trees are
added shows where extra trees stop helping - and confirms that a larger ensemble than a small
default is justified.

In [ ]:
oob_track = []
oob_params = {k: v for k, v in best.items() if k != 'n_estimators'}
for n in [100, 200, 400, 600, 800, 1000, 1200]:
    probe = RandomForestRegressor(n_estimators=n, oob_score=True, bootstrap=True,
                                  random_state=RS, n_jobs=-1, **oob_params)
    probe.fit(Xf, yf)
    oob_track.append({'trees': n, 'oob_r2': round(probe.oob_score_, 4)})
oob_df = pd.DataFrame(oob_track)
print(oob_df.to_string(index=False))
fig, ax = plt.subplots(figsize=(8.4, 4.4))
ax.plot(oob_df['trees'], oob_df['oob_r2'], marker='o', color=TONE['forest'], lw=2)
ax.axvline(best['n_estimators'], color=TONE['clay'], ls='--',
           label=f"chosen: {best['n_estimators']} trees")
ax.set_title('Out-of-bag R^2 versus number of trees')
ax.set_xlabel('Trees in the forest'); ax.set_ylabel('OOB R^2'); ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### Step 17c - Final forest and forecasts
The tuned configuration is refit with out-of-bag scoring enabled. The recursive forecast refills
the load lags with the model's own predictions once past the origin (temperature and holiday stay
observed), so it is a genuine two-year forecast comparable to SARIMA/SARIMAX. A one-step
walk-forward that consumes the true previous week is reported only as a reference.

In [ ]:
forest = RandomForestRegressor(oob_score=True, bootstrap=True, random_state=RS, n_jobs=-1, **best)
forest.fit(Xf, yf)
print('Final forest:', best['n_estimators'], 'trees | OOB R^2 =', round(forest.oob_score_, 4))

# (a) one-step walk-forward: each week uses the ACTUAL previous-week load.
rf_walk = pd.Series(forest.predict(Xo), index=yo.index)

# (b) recursive multi-step from the origin, with a tree-spread prediction interval.
line_idx = mw_week.index
origin = fit_w.size
past = list(mw_week.iloc[:origin].to_numpy())
temp_col = driver['temp']
hol_col = driver['holiday']
point, band_lo, band_hi = [], [], []
for k in range(LEAD):
    pos = origin + k
    when = line_idx[pos]
    row = {
        'woy': int(when.isocalendar()[1]),
        'month': when.month,
        'temp': temp_col.iloc[pos],
        'temp_sq': temp_col.iloc[pos] ** 2,
        'temp_back': temp_col.iloc[pos - 1],
        'temp_roll4': temp_col.iloc[pos - 3:pos + 1].mean(),
        'holiday': hol_col.iloc[pos],
        'load_back1': past[pos - 1],
        'load_back52': past[pos - 52],
    }
    xr = pd.DataFrame([row])[PREDICTORS].to_numpy()
    per_tree = np.array([est.predict(xr)[0] for est in forest.estimators_])
    yhat = float(per_tree.mean())
    point.append(yhat)
    band_lo.append(float(np.percentile(per_tree, 5)))
    band_hi.append(float(np.percentile(per_tree, 95)))
    past.append(yhat)
rf_recur = pd.Series(point, index=oos_idx)
rf_lo = pd.Series(band_lo, index=oos_idx)
rf_hi = pd.Series(band_hi, index=oos_idx)
walk_err = grade(yo, rf_walk)
recur_err = grade(oos_w, rf_recur)
print(f"RF walk-forward RMSE={walk_err['RMSE']:8.1f}  MAE={walk_err['MAE']:8.1f}  MAPE={walk_err['MAPE']:5.2f}%  (actual lag-1)")
print(f"RF recursive    RMSE={recur_err['RMSE']:8.1f}  MAE={recur_err['MAE']:8.1f}  MAPE={recur_err['MAPE']:5.2f}%  (true 2-yr)")

In [ ]:
fig, ax = plt.subplots(figsize=(12.2, 4.8))
ax.plot(feat_fit.index[-70:], yf.iloc[-70:], color=TONE['ash'], lw=1.1, label='Recent history')
ax.plot(oos_idx, oos_w, color=TONE['forest'], lw=2.3, label='Actual')
ax.plot(oos_idx, rf_recur, color=TONE['plum'], lw=2.0, label='RF recursive (multi-step)')
ax.fill_between(oos_idx, rf_lo, rf_hi, color=TONE['plum'], alpha=0.18,
                label='Tree-spread 5-95% band')
ax.plot(oos_idx, rf_walk, color=TONE['clay'], lw=1.3, ls=(0, (2, 2)),
        label='RF walk-forward (actual lag-1)')
ax.set_title('Random Forest forecasts with tree-spread interval')
ax.set_ylabel('MW'); ax.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

### Step 17d - Feature relevance, read two ways
Impurity importance is fast but biased toward high-cardinality features; permutation importance
on the hold-out is slower but model-agnostic. Showing both guards against a misleading single
ranking.

In [ ]:
imp_gini = pd.Series(forest.feature_importances_, index=PREDICTORS).sort_values()
perm = permutation_importance(forest, Xo, yo, n_repeats=25, random_state=RS,
                              scoring='neg_root_mean_squared_error')
imp_perm = pd.Series(perm.importances_mean, index=PREDICTORS).loc[imp_gini.index]
fig, (pa, pb) = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
pa.barh(imp_gini.index, imp_gini.values, color=TONE['forest'], alpha=0.85)
pa.set_title('Impurity importance'); pa.set_xlabel('Mean impurity decrease')
pb.barh(imp_perm.index, imp_perm.values, color=TONE['steel'], alpha=0.85,
        xerr=perm.importances_std[[PREDICTORS.index(f) for f in imp_gini.index]],
        error_kw=dict(ecolor=TONE['ash']))
pb.set_title('Permutation importance (hold-out)'); pb.set_xlabel('RMSE increase when shuffled')
plt.tight_layout()
plt.show()

## Step 18 - Hourly LSTM

### Literature review
The Long Short-Term Memory cell (Hochreiter & Schmidhuber, 1997) adds input, forget and output
gates to a recurrent network, letting it carry information across long spans without the
vanishing gradients that hamper plain RNNs. That property has made LSTMs a mainstay of
short-term electricity-load forecasting. Kong et al. (2019, *IEEE Transactions on Smart Grid*)
report an LSTM beating classical baselines on residential load by learning short-run dynamics
and daily/seasonal shape directly from the raw signal. Marino et al. (2016, *IECON*) apply
standard and sequence-to-sequence LSTMs to building demand; Shi et al. (2018, *IEEE Transactions
on Smart Grid*) curb the volatility of individual households with a pooling-based deep RNN; and
Bouktif et al. (2018, *Energies*) tune LSTM depth and look-back with feature selection and a
genetic algorithm. The recurring caveats are a hunger for data and error accumulation in long
recursive forecasts - both quantified below by contrasting a rolling one-step evaluation with a
genuine open-loop multi-step forecast.

**References**
- Hochreiter, S. & Schmidhuber, J. (1997). Long short-term memory. *Neural Computation*, 9(8), 1735-1780.
- Kong, W., Dong, Z. Y., Jia, Y., Hill, D. J., Xu, Y. & Zhang, Y. (2019). Short-term residential load forecasting based on LSTM recurrent neural network. *IEEE Transactions on Smart Grid*, 10(1), 841-851.
- Marino, D. L., Amarasinghe, K. & Manic, M. (2016). Building energy load forecasting using deep neural networks. *IECON 2016*, 7046-7051.
- Shi, H., Xu, M. & Li, R. (2018). Deep learning for household load forecasting - a novel pooling deep RNN. *IEEE Transactions on Smart Grid*, 9(5), 5271-5280.
- Bouktif, S., Fiaz, A., Ouni, A. & Serhani, M. A. (2018). Optimal deep learning LSTM model for electric load forecasting using feature selection and genetic algorithm. *Energies*, 11(7), 1636.


### Step 18a - Sequence preparation (no scaler leakage)
The hourly series is scaled with a MinMax scaler **fit on the training span only**, then a
168-hour (one-week) sliding window is built.

In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler
tf.keras.utils.set_random_seed(RS)
raw_hz = mw_hour.to_numpy()
HRS = 24 * 7 * LEAD                      # two-year hold-out in hours
fit_hrs = len(raw_hz) - HRS
# Fit the scaler on the training span only; fitting on the full series would leak the
# hold-out min/max into the training inputs.
scaler_h = MinMaxScaler()
scaler_h.fit(raw_hz[:fit_hrs].reshape(-1, 1))
norm_hz = scaler_h.transform(raw_hz.reshape(-1, 1)).ravel()
WIN = 168
seq_X = np.stack([norm_hz[i - WIN:i] for i in range(WIN, len(norm_hz))])[:, :, None]
seq_y = norm_hz[WIN:]
cut = len(seq_X) - HRS
Xtr_h, ytr_h = seq_X[:cut], seq_y[:cut]
Xho_h, yho_h = seq_X[cut:], seq_y[cut:]
print('LSTM tensors -> train', Xtr_h.shape, '| hold-out', Xho_h.shape)

### Step 18b - Architecture sweep and a pinned final design
Five recurrent designs are compared by validation loss with early stopping. To keep results
reproducible, the final architecture is **pinned** (not the per-run minimum), and the final head
is a **linear `Dense(1)`** so the recursion stays numerically stable.

In [ ]:
def assemble(widths, drop):
    net = Sequential()
    net.add(Input(shape=(WIN, 1)))
    for j, w in enumerate(widths):
        net.add(LSTM(w, return_sequences=(j < len(widths) - 1)))
        net.add(Dropout(drop))
    net.add(Dense(1, activation='linear'))   # linear head - stable under recursion
    net.compile(optimizer='adam', loss='mse')
    return net

nn_grid = [
    {'tag': 'compact-56', 'widths': [56], 'drop': 0.10, 'batch': 112},
    {'tag': 'twin-88-44', 'widths': [88, 44], 'drop': 0.20, 'batch': 112},
    {'tag': 'twin-112-56', 'widths': [112, 56], 'drop': 0.25, 'batch': 64},
    {'tag': 'broad-72', 'widths': [72], 'drop': 0.20, 'batch': 96},
    {'tag': 'triple-120-60-30', 'widths': [120, 60, 30], 'drop': 0.30, 'batch': 48},
]
board = []
for spec in nn_grid:
    tf.random.set_seed(RS)
    trial = assemble(spec['widths'], spec['drop'])
    stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    hist = trial.fit(Xtr_h, ytr_h, epochs=10, batch_size=spec['batch'],
                     validation_split=0.1, callbacks=[stop], verbose=0)
    board.append({'design': spec['tag'], 'val_loss': round(min(hist.history['val_loss']), 6)})
board_df = pd.DataFrame(board).sort_values('val_loss', ignore_index=True)
print(board_df.to_string(index=False))
final_spec = {'tag': 'twin-112-56', 'widths': [112, 56], 'drop': 0.25, 'batch': 64}
print('\nValidation leader :', board_df.iloc[0]['design'])
print('Pinned architecture:', final_spec['tag'])

In [ ]:
tf.random.set_seed(RS)
lstm = assemble(final_spec['widths'], final_spec['drop'])
stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
curve = lstm.fit(Xtr_h, ytr_h, epochs=10, batch_size=final_spec['batch'],
                 validation_split=0.1, callbacks=[stop], verbose=0)
fig, ax = plt.subplots(figsize=(9.2, 4.0))
ax.plot(curve.history['loss'], color=TONE['forest'], marker='o', ms=3, label='Train')
ax.plot(curve.history['val_loss'], color=TONE['clay'], marker='s', ms=3, label='Validation')
ax.set_title('LSTM training curve'); ax.set_xlabel('Epoch'); ax.set_ylabel('MSE (scaled)')
ax.legend()
plt.tight_layout()
plt.show()

### Step 18c - Rolling vs open-loop evaluation
The **rolling** forecast uses the genuine previous 168 hours at each step (a one-step reference
that sees actuals). The **open-loop** forecast is the honest multi-step test: it feeds on its own
predictions, with each fed-back value **clipped to the trained range** to keep the recursion
bounded.

In [ ]:
roll = scaler_h.inverse_transform(lstm.predict(Xho_h, batch_size=256, verbose=0)).ravel()
truth = scaler_h.inverse_transform(yho_h.reshape(-1, 1)).ravel()
buf = norm_hz[cut:cut + WIN].copy()
chain = []
for stepi in range(HRS):
    nxt = lstm.predict(buf.reshape(1, WIN, 1), verbose=0)[0, 0]
    nxt = float(np.clip(nxt, 0.0, 1.0))    # keep the fed-back value inside the trained range
    chain.append(nxt)
    buf = np.concatenate([buf[1:], [nxt]])
    if (stepi + 1) % 2500 == 0:
        print(f'   open-loop {stepi + 1}/{HRS}')
chain = scaler_h.inverse_transform(np.array(chain).reshape(-1, 1)).ravel()
hz_idx = de_load.index[WIN + cut:]
roll_wk = pd.Series(roll, index=hz_idx).resample('W').mean()
truth_wk = pd.Series(truth, index=hz_idx).resample('W').mean()
chain_wk = pd.Series(chain, index=hz_idx[:HRS]).resample('W').mean()
chain_truth_wk = pd.Series(truth[:HRS], index=hz_idx[:HRS]).resample('W').mean()
print(f'Rolling   RMSE weekly : {rmse_val(truth_wk, roll_wk):8.1f} MW  [one-step, sees actuals]')
print(f'Rolling   RMSE hourly : {rmse_val(truth, roll):8.1f} MW  [one-step, sees actuals]')
print(f'Open-loop RMSE weekly : {rmse_val(chain_truth_wk, chain_wk):8.1f} MW  [true multi-step]')
print(f'Open-loop RMSE hourly : {rmse_val(truth[:HRS], chain):8.1f} MW  [true multi-step]')

## Step 19 - Model scorecard
Every model on the shared hold-out, tagged by forecast type so one-step references are never
compared against genuine multi-step forecasts.

In [ ]:
def card(tag, actual, guess, kind):
    e = grade(actual, guess)
    return {'model': tag, 'kind': kind, 'rmse_mw': round(e['RMSE'], 1),
            'mae_mw': round(e['MAE'], 1), 'mape_pct': round(e['MAPE'], 2)}
board5 = pd.DataFrame([
    card('Mean', oos_w, naive_set['Mean'], 'multi-step'),
    card('Naive', oos_w, naive_set['Naive'], 'multi-step'),
    card('Seasonal naive', oos_w, naive_set['Seasonal naive'], 'multi-step'),
    card('Drift', oos_w, naive_set['Drift'], 'multi-step'),
    card('SARIMA', oos_w, sarima_fc, 'multi-step'),
    card('SARIMAX', oos_w, sarimax_fc, 'multi-step (conditional)'),
    card('Random Forest recursive', oos_w, rf_recur, 'multi-step (conditional)'),
    card('Random Forest walk-forward', yo, rf_walk, 'one-step (actual lag-1)'),
    card('LSTM open-loop', chain_truth_wk, chain_wk, 'multi-step'),
    card('LSTM rolling', truth_wk, roll_wk, 'one-step (actual lag-1)'),
]).sort_values('rmse_mw', ignore_index=True)
anchor = board5.loc[board5['model'] == 'Seasonal naive', 'rmse_mw'].iloc[0]
board5['delta_vs_naive'] = (board5['rmse_mw'] - anchor).round(1)
print('Seasonal-naive weekly RMSE =', anchor, 'MW\n')
print(board5.to_string(index=False))

In [ ]:
multi = board5[board5['kind'].str.startswith('multi-step')].sort_values('rmse_mw')
shade = [TONE['forest'] if v == multi['rmse_mw'].min() else TONE['ash'] for v in multi['rmse_mw']]
fig, ax = plt.subplots(figsize=(11, 5.2))
ax.barh(multi['model'], multi['rmse_mw'], color=shade, alpha=0.88)
ax.axvline(anchor, color=TONE['clay'], ls='--', lw=1.3, label=f'Seasonal naive ({anchor:.0f} MW)')
for y, v in zip(range(len(multi)), multi['rmse_mw']):
    ax.text(v, y, f' {v:.0f}', va='center', fontsize=8)
ax.invert_yaxis()
ax.set_title('Multi-step weekly RMSE (lower is better)')
ax.set_xlabel('RMSE [MW]'); ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15.5, 6.2))
ax.plot(fit_w.index, fit_w, color=TONE['ash'], lw=0.8, alpha=0.7, label='Train')
ax.plot(oos_idx, oos_w, color=TONE['forest'], lw=2.4, label='Actual (hold-out)')
ax.plot(oos_idx, naive_set['Seasonal naive'], color=TONE['sand'], lw=1.4, ls=(0, (5, 2)), label='Seasonal naive')
ax.plot(oos_idx, sarima_fc, color=TONE['steel'], lw=1.4, label='SARIMA')
ax.plot(oos_idx, sarimax_fc, color=TONE['clay'], lw=1.4, label='SARIMAX')
ax.plot(oos_idx, rf_recur, color=TONE['plum'], lw=2.2, label='Random Forest recursive')
lo = min(fit_w.min(), oos_w.min()) * 0.9
hi = max(fit_w.max(), oos_w.max()) * 1.1
ax.set_ylim(lo, hi)
ax.set_title('Multi-step forecasts vs actual (open-loop LSTM shown separately)')
ax.set_ylabel('MW'); ax.legend(ncol=3, fontsize=8, loc='lower left')
plt.tight_layout()
plt.show()
fig, ax = plt.subplots(figsize=(15.5, 3.6))
ax.plot(oos_idx, oos_w, color=TONE['forest'], lw=2.2, label='Actual (hold-out)')
ax.plot(chain_wk.index, chain_wk, color=TONE['plum'], lw=1.6, label='LSTM open-loop (recursive)')
ax.set_title('Open-loop LSTM over the two-year horizon (bounded by clipping)')
ax.set_ylabel('MW'); ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

## Step 20 - Analytical questions

**Q1 - Which models meaningfully beat the seasonal-naive benchmark?**
Compared strictly within a forecast type, no multi-step model improves on 'repeat last year' by
a decisive margin: the recursive Random Forest and SARIMAX come closest, at best drawing level
with the seasonal naive within run-to-run noise, while SARIMA and the naive baselines trail it
and the open-loop LSTM drifts far off through error accumulation. German weekly demand is
governed by a near-constant annual cycle, so replaying last year is an exceptionally strong
two-year baseline, and the hold-out contains the 2020 COVID-19 dip - an exogenous shock no model
anticipates - which further protects the naive replay. The walk-forward Random Forest and rolling
LSTM look sharper only because they consume the true previous value; they are one-step references
and are not comparable to the multi-step forecasts.

**Q2 - How was leakage avoided in the Random Forest?**
Every engineered feature looks strictly backward: `load_back1 = shift(1)`, `load_back52 =
shift(52)`, `temp_back = shift(1)` and `temp_roll4` is a trailing four-week mean. The
hyper-parameter search uses `TimeSeriesSplit`, so each validation fold is later than its training
fold. The recursive forecast refills its load lags with the model's own predictions once past the
origin, so no future actual enters the multi-step figures; temperature and the holiday flag are
used only under the stated conditional assumption.

**Q3 - Justify the differencing and seasonal orders.**
`d` is not hard-coded. Candidate orders are refit at both `d = 0` and `d = 1` (AIC is not
comparable across `d`), and the smallest `d` that yields white-noise residuals (Ljung-Box
`p > 0.05`) is chosen - consistent with the ADF/KPSS evidence and avoiding the over-differencing
of `d = 2`. `D = 1` at `s = 52` follows from the dominant annual cycle seen in the STL and ACF.
Within the chosen `d`, orders are ranked by AIC among the residual-adequate candidates with a
parsimony tie-break.

**Q4 - Why grow the forest larger, and did it help?**
A single small forest under-uses the data. The randomised search reaches for more and deeper
trees; the out-of-bag curve shows R^2 rising with tree count before flattening, which both
justifies the larger ensemble and identifies the point of diminishing returns. Deeper trees
capture the temperature/load non-linearity, while bootstrapping plus feature sub-sampling keep
variance in check.

**Q5 - Interpretability and complexity.**

| Aspect | SARIMAX | Random Forest | LSTM |
|---|---|---|---|
| Interpretability | High (coefficients, intervals) | Medium (impurity + permutation) | Low (black box) |
| Uncertainty | Native intervals | Tree-spread interval | None natively |
| Non-linearity | Manual (squared term) | Native | Native (learned) |
| Training cost | Minutes | Minutes (search) | Minutes to hours |

**Q6 - Which model for operational use?**
**SARIMAX** remains the pragmatic operational choice: native prediction intervals, interpretable
temperature/holiday coefficients, direct exogenous support and cheap retraining. The tuned Random
Forest is a strong, non-linear competitor that now carries a usable tree-spread interval, but it
still requires recursive feature construction for multi-step use; the open-loop LSTM is unsuited
to this long horizon. Any deployment should keep benchmarking against the seasonal naive and
retrain as post-shock data arrives.


## Step 21 - Summary
Across five models the study confirms a consistent story for German weekly electricity demand:
the annual cycle is so regular that a seasonal-naive replay is a formidable two-year baseline,
matched but not clearly beaten by SARIMAX and a fully tuned Random Forest, while a univariate
open-loop LSTM cannot sustain a two-year recursive forecast. Growing and tuning the forest
- more trees, deeper branches, out-of-bag guidance and a tree-spread interval - extracts more
from the same estimator family than a single fixed fit, without ever letting future information
leak into the past.